# 1. Controls and attainable information

**Question:** Are the dynamics comparisons genuinely matched, and which latent information could each sensor recover even with an ideal estimator?

Local and scrambled transitions have identical stationary observation distributions and transition spectra when sensor maps stay fixed. This notebook checks those identities and adds a privileged observation-aware reference: the conditional mean of a target given a noisy sensor measurement. It never trains an encoder on that reference.

**Storage:** computations and tables stay in RAM. No result file or browser download is produced by default. Saving this notebook in your editor can still save displayed cell outputs; disable output saving or clear outputs if you do not want them in the notebook file.

In [ ]:
from pathlib import Path
import sys
import subprocess
from zipfile import ZipFile

REPO_URL = 'https://github.com/AlexM3020/The-Role-of-Dynamical-Structure-in-Cross-Modal-Representation-Learning.git'
REPO_REF = 'main'  # For a frozen run, choose a commit before starting a fresh runtime.
SOURCE_ARCHIVE = None  # Optional uploaded source ZIP; not needed in Colab with GitHub access.
INSTALL_DEPENDENCIES = False  # Enable only if the runtime lacks compatible dependencies.
cloud = next((p for p in (Path('/content'), Path('/kaggle/working')) if p.exists()), None)
if SOURCE_ARCHIVE is not None:
    if cloud is None:
        raise RuntimeError('Use the local checkout here; source extraction is for cloud runtimes.')
    destination = cloud / 'dynalign_uploaded_source'
    with ZipFile(SOURCE_ARCHIVE) as archive:
        for name in archive.namelist():
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Unsafe archive member')
        archive.extractall(destination)
    candidates = [destination]
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'src/dynalign/revised.py').exists()), None)
if ROOT is None and SOURCE_ARCHIVE is None and cloud is not None:
    destination = cloud / 'dynalign_github'
    if not destination.exists():
        subprocess.check_call(['git', 'clone', '--quiet', REPO_URL, str(destination)])
        subprocess.check_call(['git', '-C', str(destination), 'fetch', '--quiet', 'origin', REPO_REF])
        subprocess.check_call(['git', '-C', str(destination), 'checkout', '--quiet', '--detach', 'FETCH_HEAD'])
    ROOT = destination
if ROOT is None or not (ROOT / 'src/dynalign/revised.py').exists():
    raise FileNotFoundError('Open the current checkout locally, or use a fresh Colab/Kaggle runtime with Internet enabled.')
# Existing checkouts are deliberately reused, not reset or silently updated.
# Restart in a fresh runtime to switch REPO_REF or load a newer GitHub revision.
if 'dynalign' in sys.modules and Path(sys.modules['dynalign'].__file__).resolve().parents[2] != ROOT.resolve():
    raise RuntimeError('Another source version is already imported; restart the runtime before switching.')
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT)])
sys.path.insert(0, str(ROOT / 'src'))
revision = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], capture_output=True, text=True)
print('Source revision:', revision.stdout.strip() if revision.returncode == 0 else 'uploaded source bundle')
print('Source setup only: result saving remains disabled by default.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from hashlib import sha256
from dynalign.revised import attainable_reference
from dynalign.worlds import transition_family, diagnostics, spectral_embedding, sensor_table, sample_pairs, circle_coordinates, observation_roughness
from dynalign.metrics import linear_cka, procrustes_test_error

## Experiments to try

1. **Construction audit:** compare local and scrambled kernels with fixed sensors. Confirm stationarity, entropy, spectrum and the distinction between mixing gap and embedding gap.
2. **Coordinate-change control:** co-permute both sensors and dynamics. Observation pairs should be identical after relabeling; this is not the main intervention.
3. **Geometry sanity:** compare rotations with invertible anisotropic transforms. Information preservation does not imply matching geometry.
4. **Attainable information:** compare full-state and doubled-angle quotient targets at noise 0, 0.05, and 0.30; repeat with an aliased sensor. The conditional mean uses the known generative model and is a privileged Bayes-MSE reference, not a learned result or a finite-sample confidence bound.
5. **Boundary check:** try an odd embedding dimension and inspect a zero boundary eigengap. An arbitrary basis then does not define a uniquely identified target.

Edit the values below and rerun subsequent cells. No expensive training occurs here.

In [ ]:
# All controls for this workflow are editable here.
N_STATES = 32
EMBEDDING_DIM = 4
SEED = 17
SENSOR_WIDTH = 12
NOISE_LEVELS = [0.0, 0.05, 0.30]
REFERENCE_REPEATS = 16  # Increase for more precise Monte Carlo risk estimates.
SAVE_RESULTS = False
OUTPUT_DIR = None  # Only used if SAVE_RESULTS=True; no laptop path is inferred.

family, permutation = transition_family(N_STATES, SEED)
table_a = sensor_table(N_STATES, SEED + 1, width=SENSOR_WIDTH)
table_b = sensor_table(N_STATES, SEED + 2, width=SENSOR_WIDTH)
if SAVE_RESULTS and OUTPUT_DIR is None:
    raise ValueError('Choose an explicit destination before enabling saving.')
print('Configured stationary controls and observation-aware references; storage is opt-in.')

## Verify the intervention before interpreting learning

A permutation-conjugated transition changes adjacency relative to the **unchanged** sensor geometry; the abstract chain and its full spectrum stay the same. With a uniform stationary state and fixed conditionally independent observation channels, the instantaneous joint observation law is unchanged. No differential-entropy estimator is needed for that statement.

The IID control shares the stationary marginal but not the transition entropy or a uniquely selected spectral subspace. Any zero boundary gap is reported explicitly.

In [ ]:
audit = pd.DataFrame([{'condition': name, **diagnostics(p, EMBEDDING_DIM),
                       'roughness_a': observation_roughness(p, table_a),
                       'roughness_b': observation_roughness(p, table_b)}
                      for name, p in family.items()]).set_index('condition')
np.testing.assert_allclose(np.linalg.eigvalsh(family['local']), np.linalg.eigvalsh(family['scrambled']), atol=1e-12)
for key in ['stationary_max_error', 'transition_entropy_nats', 'lag1_latent_mi_nats', 'mixing_gap', 'embedding_gap']:
    np.testing.assert_allclose(audit.loc['local', key], audit.loc['scrambled', key], atol=1e-12)
pairs = sample_pairs(family['local'], 256, SEED + 100)
inverse = np.argsort(permutation)
for table in (table_a, table_b):
    np.testing.assert_array_equal(table[pairs], table[permutation][inverse[pairs]])
h, gap = spectral_embedding(family['local'], EMBEDDING_DIM)
print('Unique selected spectral subspace:', gap > 1e-8)
print('Co-permutation observation identity: passed')
display(audit)

In [ ]:
reference_rows = []
for aliased in [False, True]:
    table = sensor_table(N_STATES, SEED + 2, aliased=aliased, width=SENSOR_WIDTH)
    targets = {'coordinate': circle_coordinates(N_STATES),
               'quotient': circle_coordinates(N_STATES, harmonic=2)}
    for condition in ['local', 'scrambled']:
        basis, boundary_gap = spectral_embedding(family[condition], EMBEDDING_DIM)
        available_targets = dict(targets)
        if boundary_gap > 1e-8:
            available_targets['spectral'] = basis
        for noise in NOISE_LEVELS:
            for name, target in available_targets.items():
                values = attainable_reference(table, noise, target, REFERENCE_REPEATS, SEED + 500)
                reference_rows.append({'aliased': aliased, 'condition': condition,
                                       'noise': noise, 'target': name, **values})
references = pd.DataFrame(reference_rows)
display(references)
rng = np.random.default_rng(SEED)
x, xt = rng.normal(size=(128, EMBEDDING_DIM)), rng.normal(size=(128, EMBEDDING_DIM))
q, _ = np.linalg.qr(rng.normal(size=(EMBEDDING_DIM, EMBEDDING_DIM)))
warp = np.diag(np.geomspace(1, 10, EMBEDDING_DIM))
print('Rotation CKA:', linear_cka(xt, xt @ q))
print('Invertible anisotropic CKA:', linear_cka(xt, xt @ warp))
assert procrustes_test_error(x, x @ q, xt, xt @ q) < 1e-15

## Interpret the reference correctly

The known Gaussian-mixture posterior gives a conditional-mean estimator that minimizes expected squared error for the specified target. Its Monte Carlo score is estimated from finitely many noisy measurements: it is not an exact empirical ceiling for every finite-sample metric or a guarantee of linear decodability.

At exact antipodal aliasing, full circular coordinates have conditional mean zero, while the doubled-angle quotient remains accessible. Noise can reduce attainable recovery even when the clean transition eigenspace is unique. This is why a low clean-state spectral probe score alone cannot diagnose training failure.

The spectrum plot below is a construction diagnostic, not a result of neural training.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for name, transition in family.items():
    axes[0].plot(np.linalg.eigvalsh(transition)[::-1], '.-', label=name)
axes[0].set(title='Transition spectra', xlabel='Eigenvalue index')
axes[0].legend()
for (aliased, target), block in references[references.condition == 'local'].groupby(['aliased', 'target']):
    axes[1].plot(block.noise, block.bayes_mc_r2, 'o-', label=f'{target}, alias={aliased}')
axes[1].set(title='Observation-aware conditional-mean reference', xlabel='Noise standard deviation', ylabel='Monte Carlo R²')
axes[1].legend(fontsize=7)
fig.tight_layout()
plt.show()  # Display only; no savefig call.
control_tables = {'controls': audit.reset_index(), 'attainable_references': references}

## Optional export — no automatic laptop files

The next cell creates ZIP bytes in RAM, not on disk. Set `SAVE_RESULTS=True` and explicitly choose `OUTPUT_DIR` only if you want a file saved in that runtime. In Colab this can be a mounted Drive path; in Kaggle it can be a deliberate working/output path. Neither is chosen automatically.

`SHOW_DOWNLOAD_LINK=True` displays a link that downloads **only when you click it**. A click may save to your laptop. Leaving the link disabled and saving disabled produces no result files. Notebook editors can still save displayed output inside the notebook; that is separate from this export policy.

In [ ]:
from io import BytesIO
from zipfile import ZIP_DEFLATED
import json
SHOW_DOWNLOAD_LINK = False  # A displayed link still requires a manual click.
buffer = BytesIO()
with ZipFile(buffer, 'w', ZIP_DEFLATED) as archive:
    for name, table in control_tables.items():
        archive.writestr(name + '.csv', table.to_csv(index=False))
    archive.writestr('configuration.json', json.dumps({'n_states': N_STATES, 'dim': EMBEDDING_DIM,
                       'seed': SEED, 'sensor_width': SENSOR_WIDTH, 'noise_levels': NOISE_LEVELS,
                       'reference_repeats': REFERENCE_REPEATS}, indent=2))
EXPORT_BYTES = buffer.getvalue()
if SAVE_RESULTS:
    target = Path(OUTPUT_DIR)
    target.mkdir(parents=True, exist_ok=True)
    filename = target / ('controls-' + sha256(EXPORT_BYTES).hexdigest()[:16] + '.zip')
    filename.write_bytes(EXPORT_BYTES)
    print('Explicitly saved to:', filename)
else:
    print(f'{len(EXPORT_BYTES):,} export bytes in RAM; no result file written.')
if SHOW_DOWNLOAD_LINK:
    import base64
    from IPython.display import HTML
    display(HTML('<a download="control_audit.zip" href="data:application/zip;base64,' +
                 base64.b64encode(EXPORT_BYTES).decode() + '">Click to download</a>'))